# CSI 300 vs CSI 500 — Alpha 对比

对比 Top 5 策略在大市值（CSI 300）vs 中盘（CSI 500）股票池的表现，验证中盘股是否提供更高 alpha。

**策略池**：reversed_gtja_vwap, gtja_vwap, gtja_momentum, gtja_volume_price, gtja_volatility

**股票池**：
- CSI 300：default.yaml 中的 100 只大市值股
- CSI 500：csi500.yaml 中的 ~500 只中盘股

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use('Agg')
from pathlib import Path

import matplotlib.pyplot as plt

from analysis.pool_matrix import pivot_matrix, run_matrix
from config.loader import load_config
from data import validate_ohlcv
from data.fetcher import fetch_daily_batch
from data.filters import detect_limit_price, detect_suspension
from data.universe import resolve_universe

print('All imports OK')

## Step 1: 加载 CSI 300 数据

In [ ]:
cfg_300 = load_config(Path('../configs/default.yaml'))
universe_300 = cfg_300.get('universe', {})

STOCKS_300 = resolve_universe(universe_300)
START = universe_300.get('start_date', '2023-01-01')
END = universe_300.get('end_date', '2026-05-23')
RAW_DIR = Path('../data/raw')

print(f'Loading {len(STOCKS_300)} CSI 300 stocks from {START} to {END}...')
data_300 = fetch_daily_batch(STOCKS_300, START, END, RAW_DIR, sleep_sec=0.2, progress=True)
data_300 = detect_limit_price(data_300)
data_300 = detect_suspension(data_300)
validate_ohlcv(data_300)

print(f'CSI 300 data: {len(data_300)} rows, {data_300["code"].nunique()} stocks')
print(f'Date range: {data_300["date"].min().date()} ~ {data_300["date"].max().date()}')

## Step 2: 加载 CSI 500 数据

In [ ]:
cfg_500 = load_config(Path('../configs/csi500.yaml'))
universe_500 = cfg_500.get('universe', {})

STOCKS_500 = resolve_universe(universe_500)
START_500 = universe_500.get('start_date', '2023-01-01')
END_500 = universe_500.get('end_date', '2026-05-23')

print(f'Loading {len(STOCKS_500)} CSI 500 stocks from {START_500} to {END_500}...')
print('First run may take ~2min for uncached stocks. Subsequent runs use parquet cache.')
data_500 = fetch_daily_batch(STOCKS_500, START_500, END_500, RAW_DIR, sleep_sec=0.2, progress=True)
data_500 = detect_limit_price(data_500)
data_500 = detect_suspension(data_500)
validate_ohlcv(data_500)

print(f'CSI 500 data: {len(data_500)} rows, {data_500["code"].nunique()} stocks')
print(f'Date range: {data_500["date"].min().date()} ~ {data_500["date"].max().date()}')

## Step 3: 定义策略

In [ ]:
STRATEGY_SPECS = [
    {'name': 'reversed_gtja_vwap'},
    {'name': 'gtja_vwap'},
    {'name': 'gtja_momentum'},
    {'name': 'gtja_volume_price'},
    {'name': 'gtja_volatility'},
]

print('Strategies:')
for s in STRATEGY_SPECS:
    print(f'  - {s["name"]}')

## Step 4: 运行矩阵回测

In [ ]:
pool_300 = {'CSI300': STOCKS_300}
pool_500 = {'CSI500': STOCKS_500}

print('Running CSI 300 matrix...')
results_300 = run_matrix(pool_300, STRATEGY_SPECS, data_300)
print(f'  Done: {len(results_300)} rows')

print('Running CSI 500 matrix...')
results_500 = run_matrix(pool_500, STRATEGY_SPECS, data_500)
print(f'  Done: {len(results_500)} rows')

## Step 5: 对比表

In [ ]:
# 合并两个 universe 的结果
comparison = pd.concat([results_300, results_500], ignore_index=True)

# Sharpe 对比
sharpe_pivot = pivot_matrix(comparison, metric='sharpe_ratio')
print('=== Sharpe Ratio ===')
sharpe_pivot

In [ ]:
# 构建详细对比表
detail = comparison[['strategy', 'pool', 'total_return', 'annual_return', 'sharpe_ratio', 'max_drawdown', 'win_rate']].copy()
detail = detail.rename(columns={
    'pool': 'universe',
    'total_return': 'return',
    'annual_return': 'annual',
    'sharpe_ratio': 'sharpe',
    'max_drawdown': 'max_dd',
    'win_rate': 'win_rate',
})

# 按 strategy pivot 出双列
metrics = ['return', 'annual', 'sharpe', 'max_dd', 'win_rate']
rows = []
for strat in detail['strategy'].unique():
    row = {'strategy': strat}
    for m in metrics:
        val_300 = detail[(detail['strategy'] == strat) & (detail['universe'] == 'CSI300')][m].values
        val_500 = detail[(detail['strategy'] == strat) & (detail['universe'] == 'CSI500')][m].values
        if len(val_300) > 0:
            row[f'CSI300_{m}'] = val_300[0]
        if len(val_500) > 0:
            row[f'CSI500_{m}'] = val_500[0]
        if len(val_300) > 0 and len(val_500) > 0:
            row[f'delta_{m}'] = val_500[0] - val_300[0]
    rows.append(row)

detail_df = pd.DataFrame(rows)
print('=== Detailed Comparison ===')
detail_df

## Step 6: Sharpe 对比柱状图

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

strategies = detail_df['strategy'].tolist()
x = np.arange(len(strategies))
width = 0.35

sharpe_300 = detail_df['CSI300_sharpe'].tolist()
sharpe_500 = detail_df['CSI500_sharpe'].tolist()

bars1 = ax.bar(x - width/2, sharpe_300, width, label='CSI 300', color='steelblue')
bars2 = ax.bar(x + width/2, sharpe_500, width, label='CSI 500', color='coral')

ax.set_xlabel('Strategy')
ax.set_ylabel('Sharpe Ratio')
ax.set_title('CSI 300 vs CSI 500 — Sharpe Ratio')
ax.set_xticks(x)
ax.set_xticklabels(strategies, rotation=30, ha='right')
ax.legend()
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)
fig.savefig(output_dir / 'csi300_vs_csi500_sharpe.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: csi300_vs_csi500_sharpe.png')

## Step 7: 总收益 + 最大回撤对比

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 总收益对比
ax = axes[0]
ret_300 = detail_df['CSI300_return'].tolist()
ret_500 = detail_df['CSI500_return'].tolist()
bars1 = ax.bar(x - width/2, ret_300, width, label='CSI 300', color='steelblue')
bars2 = ax.bar(x + width/2, ret_500, width, label='CSI 500', color='coral')
ax.set_title('Total Return')
ax.set_xticks(x)
ax.set_xticklabels(strategies, rotation=30, ha='right')
ax.legend()
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
ax.grid(axis='y', alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# 最大回撤对比
ax = axes[1]
dd_300 = detail_df['CSI300_max_dd'].tolist()
dd_500 = detail_df['CSI500_max_dd'].tolist()
bars1 = ax.bar(x - width/2, dd_300, width, label='CSI 300', color='steelblue')
bars2 = ax.bar(x + width/2, dd_500, width, label='CSI 500', color='coral')
ax.set_title('Max Drawdown')
ax.set_xticks(x)
ax.set_xticklabels(strategies, rotation=30, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

fig.tight_layout()
fig.savefig(output_dir / 'csi300_vs_csi500_return_dd.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: csi300_vs_csi500_return_dd.png')

## Step 8: 总结

In [ ]:
print('=' * 60)
print('CSI 300 vs CSI 500 — Summary')
print('=' * 60)
print()
print(f'CSI 300 stocks: {len(STOCKS_300)}')
print(f'CSI 500 stocks: {len(STOCKS_500)}')
print()

print(f'{"Strategy":<25} {"CSI300 Sharpe":>14} {"CSI500 Sharpe":>14} {"Delta":>8}')
print('-' * 65)
for _, row in detail_df.iterrows():
    s300 = row.get('CSI300_sharpe', float('nan'))
    s500 = row.get('CSI500_sharpe', float('nan'))
    delta = row.get('delta_sharpe', float('nan'))
    better = '<<' if delta > 0.05 else ('' if abs(delta) <= 0.05 else '')
    print(f'{row["strategy"]:<25} {s300:>14.3f} {s500:>14.3f} {delta:>+8.3f} {better}')

print()
avg_delta = detail_df['delta_sharpe'].mean()
print(f'Average Sharpe delta (CSI500 - CSI300): {avg_delta:+.3f}')
if avg_delta > 0.05:
    print('CSI 500 provides meaningfully higher alpha.')
elif avg_delta < -0.05:
    print('CSI 300 outperforms CSI 500. Mid-cap expansion not beneficial.')
else:
    print('No significant difference. Alpha does not scale with market cap.')

In [ ]:
# 流动性对比
print('=== Liquidity Comparison ===')
avg_vol_300 = data_300.groupby('code')['volume'].mean().mean()
avg_vol_500 = data_500.groupby('code')['volume'].mean().mean()
print(f'CSI 300 avg daily volume: {avg_vol_300:,.0f}')
print(f'CSI 500 avg daily volume: {avg_vol_500:,.0f}')
print(f'Ratio (CSI500/CSI300): {avg_vol_500/avg_vol_300:.2f}')